# Maritime Traffic & Port Congestion Pipeline — progress demo

**DAT535 Data Engineering · Group 10** · Lucas Galvão Freitas, Victor Peugnet

This notebook shows what the pipeline does so far, running on the real data on our VM.
It grows with the project: one section per layer.

| Layer | Status (27 Oct 2026) |
|---|---|
| Foundation (package, config, VM, History Server) | ✅ done |
| **Bronze** (download, raw contract, Delta ingest) | ✅ done, tested on a real day |
| Bronze ingestion log + idempotency by file hash | ✅ done |
| Silver (cleaning, quality metrics, vessel/port model) | 🔜 planned |
| Gold (port calls, hourly congestion) + dashboard | 🔜 planned |
| Automated runs (GitHub Actions on the VM) | 🔜 planned |

## 1. Architecture

```
DMA archive (1 zip/day, ~650 MB, ~18.7 M rows)
      │  ais-pipeline download      (retries, atomic rename, skip if present)
      ▼
landing/date=YYYY-MM-DD/aisdk-YYYY-MM-DD.zip
      │  ais-pipeline bronze        (header checked against the raw contract)
      ▼
bronze.ais_raw      Delta, all strings + lineage, partition _source_date
      │  ais-pipeline silver        (types, reject codes, dedup, flags, SCD2, zones)
      ▼
silver.vessel_position · silver.dim_vessel · silver.dim_port · silver.rejects
      │  ais-pipeline gold          (quality gates first)
      ▼
gold.port_calls · gold.port_congestion_hourly   ──▶  Streamlit dashboard
```

Everything runs with PySpark 3.5 + Delta Lake on one VM (`local[4]`, 4 vCPU, 8 GB RAM),
orchestrated by GitHub Actions on a self-hosted runner.

## 2. Setup

The notebook uses the project package, so it runs exactly the same code as the pipeline.

In [ ]:
import json
import zipfile
from datetime import date

import matplotlib.pyplot as plt
from pyspark.sql import functions as F

from ais_pipeline.config import load_settings
from ais_pipeline.ingest.bronze import check_header, load_contract, read_header
from ais_pipeline.paths import landing_path, table_path
from ais_pipeline.spark import build_spark

DAY = date(2026, 9, 1)
settings = load_settings("prod")
spark = build_spark(settings)
print("Spark", spark.version, "| master", spark.sparkContext.master, "| lake", settings.lake_dir)

## 3. Source data

The Danish Maritime Authority publishes one zip per day at `http://aisdata.ais.dk/aisdk-YYYY-MM-DD.zip`,
about 3 days after the data day. Our production window is **1–28 September 2026**.

In [ ]:
zip_path = landing_path(settings, DAY) / f"aisdk-{DAY}.zip"
info = zipfile.ZipFile(zip_path).infolist()[0]
print(f"{zip_path.name}: {zip_path.stat().st_size / 1e6:,.0f} MB zipped")
print(f"{info.filename}: {info.file_size / 1e9:,.2f} GB uncompressed")

## 4. Raw data contract

Before Spark reads 3 GB, the header is checked against a versioned contract
(`contracts/ais_raw_v1.json`): 26 columns, 5 of them required. A missing required column stops
the run; a new column is kept with a warning. Column names are sanitised (`Type of mobile` →
`type_of_mobile`); values are never changed in Bronze.

In [ ]:
contract = load_contract()
member, header = read_header(zip_path)
report = check_header(header, contract)
print("status:", report.status, "| contract", report.contract_version, "| columns:", len(report.bronze_columns))
print("required:", [c["source_name"] for c in contract["columns"] if c["required"]])
print("first columns:", list(zip(header[:5], report.bronze_columns[:5])))

## 5. Bronze table

`bronze.ais_raw` is a Delta table partitioned by `_source_date`. Each day is written with
`replaceWhere`, so re-running a day **replaces** that partition instead of duplicating it.
Lineage columns record where every row came from.

In [ ]:
bronze_path = str(table_path(settings, "bronze", "ais_raw"))
bronze = spark.read.format("delta").load(bronze_path)
bronze.printSchema()

In [ ]:
detail = spark.sql(f"DESCRIBE DETAIL delta.`{bronze_path}`").select(
    "format", "partitionColumns", "numFiles", (F.col("sizeInBytes") / 1e6).alias("size_MB")
)
detail.show(truncate=False)

Delta keeps a log of every write. The `predicate` shows each write only touched its own day:

In [ ]:
spark.sql(f"DESCRIBE HISTORY delta.`{bronze_path}`").select(
    "version", "timestamp", "operation",
    F.col("operationParameters.predicate").alias("predicate"),
    F.col("operationMetrics.numOutputRows").alias("rows_written"),
).show(truncate=False)

In [ ]:
bronze.select("timestamp", "type_of_mobile", "mmsi", "latitude", "longitude", "sog",
              "ship_type", "_source_file", "_run_id").show(5, truncate=False)

### Reconciliation: every CSV line is in Bronze

In [ ]:
day_df = bronze.where(F.col("_source_date") == F.lit(str(DAY)).cast("date"))
bronze_rows = day_df.count()
with zipfile.ZipFile(zip_path) as zf, zf.open(member) as fh:
    csv_rows = sum(1 for _ in fh) - 1  # minus header
corrupt = day_df.where("_corrupt_record IS NOT NULL").count()
print(f"CSV data lines: {csv_rows:,} | Bronze rows: {bronze_rows:,} | malformed: {corrupt:,} | match: {csv_rows == bronze_rows}")

| Measure (1 Sep 2026) | Value |
|---|---|
| Download | 659 MB in 76 s |
| Bronze ingest (extract + read + write) | 97 s |
| Size on disk | zip 659 MB → CSV 3.34 GB → Bronze Delta 548 MB |

### Ingestion log and idempotency

Every Bronze attempt writes one row to `ops.ingestion_log` (`success`, `skipped` or `failed`).
Before ingesting, the pipeline compares the zip's **SHA-256** with the last `success` for that day:

- same hash → the day is already in Bronze: nothing is written, status `skipped`;
- new hash (the file changed) or `--force` → re-ingest, and `replaceWhere` replaces only that day's partition.

So the scheduled job can safely re-run any day. The cell below asks for the same day again.
The first time a day is logged it is ingested (~100 s); every later run is `skipped` in a few seconds.


In [ ]:
import time

from ais_pipeline.ingest.bronze import run_bronze_day
from ais_pipeline.ingest.ingestion_log import log_path
from ais_pipeline.ops.runs import new_run_id

started = time.monotonic()
entry = run_bronze_day(spark, settings, DAY, run_id=new_run_id())
print(f"{DAY}: {entry.status} in {time.monotonic() - started:.1f} s | {entry.message}")


In [ ]:
spark.read.format("delta").load(str(log_path(settings))).select(
    "source_date", "status", "row_count", F.substring("file_sha256", 1, 12).alias("sha256"),
    "message", "run_id", "logged_at",
).orderBy("logged_at").show(truncate=False)


## 6. First look at the data (what Silver must handle)

A preview of the profiling work (issue #16), using both the DataFrame API and Spark SQL.

In [ ]:
day_df.createOrReplaceTempView("bronze_day")
spark.sql("""
    SELECT type_of_mobile, COUNT(*) AS rows, ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM bronze_day GROUP BY type_of_mobile ORDER BY rows DESC
""").show(truncate=False)

Placeholders and sentinels that Silver turns into nulls, flags or rejects — counted in **one** pass:

In [ ]:
PLACEHOLDERS = ["Unknown", "Unknown value", "Undefined", "N/A"]
checks = day_df.agg(
    F.count(F.lit(1)).alias("rows"),
    F.sum(F.col("ship_type").isin(PLACEHOLDERS).cast("int")).alias("ship_type_placeholder"),
    F.sum(F.col("sog").isNull().cast("int")).alias("sog_null"),
    F.sum((F.col("sog").cast("double") >= 102.3).cast("int")).alias("sog_sentinel_102_3"),
    F.sum((F.col("heading") == "511").cast("int")).alias("heading_511"),
    F.sum((F.col("latitude").cast("double") == 91).cast("int")).alias("lat_91"),
    F.sum((F.length("mmsi") != 9).cast("int")).alias("mmsi_not_9_digits"),
)
checks.show(vertical=True)

In [ ]:
dups = day_df.count() - day_df.drop("_corrupt_record", "_ingest_ts", "_run_id").dropDuplicates().count()
print(f"exact duplicate rows: {dups:,} ({100 * dups / bronze_rows:.1f}% of the day)")

Messages per hour (completeness check — a full day must have all 24 hours):

In [ ]:
per_hour = (
    day_df.withColumn("hour", F.substring("timestamp", 12, 2).cast("int"))
    .groupBy("hour").count().orderBy("hour").toPandas()  # 24 rows: safe to collect
)
ax = per_hour.plot.bar(x="hour", y="count", legend=False, figsize=(10, 3), color="#3b6ea5")
ax.set_ylabel("messages"); ax.set_title(f"AIS messages per hour, {DAY}")
plt.show()

A few vessels send far more messages than others — this **skew** matters for the per-vessel window in Silver (experiment P3):

In [ ]:
day_df.groupBy("mmsi", "type_of_mobile").count().orderBy(F.desc("count")).show(10)

### Where the ships are (0.5 % sample) and our three ports

In [ ]:
PORTS = {"Esbjerg": (55.466, 8.430), "Aarhus": (56.150, 10.230), "Copenhagen": (55.705, 12.605)}
pts = (
    day_df.where("type_of_mobile IN ('Class A', 'Class B')")
    .select(F.col("latitude").cast("double").alias("lat"), F.col("longitude").cast("double").alias("lon"))
    .where("lat BETWEEN 53 AND 59 AND lon BETWEEN 3 AND 17")
    .sample(fraction=0.005, seed=42)
    .toPandas()  # ~80k points: bounded by the sample fraction
)
fig, ax = plt.subplots(figsize=(9, 7))
ax.hexbin(pts.lon, pts.lat, gridsize=150, bins="log", cmap="Blues", mincnt=1)
for name, (lat, lon) in PORTS.items():
    ax.plot(lon, lat, "o", color="#d1495b"); ax.annotate(name, (lon, lat), xytext=(5, 5), textcoords="offset points")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude"); ax.set_title(f"AIS positions, {DAY} (sample of {len(pts):,})")
plt.show()

## 7. Next steps

| When | What |
|---|---|
| 27–28 Oct | CI on every PR; test fixture and profiling report |
| 26–28 Oct | **Silver cleaning**: explicit types, reject codes, deduplication, sentinel flags, before/after quality metrics |
| 27–29 Oct | Port reference, movement features, port/anchorage zones, vessel history (SCD Type 2) |
| 29–31 Oct | Gold tables with quality gates, performance experiments, automated runs, Streamlit dashboard |
| 2 Nov | Report and submission |

**Questions for the TA**
1. Is one region (3 Danish ports) and a 4-week window enough data volume for the project?
2. Is the planned evidence for Silver (before/after quality table + rejects with reason codes) what you expect?
3. Any expectation on the performance experiments beyond before/after timings, plans and Spark UI screenshots?